# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os


# ============================================================
# Cache / Env
# ============================================================

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT


import gc
import json
import math
import random
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset




# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

EMOTIONS = ["anger", "disgust", "fear", "joy", "surprise", "sadness"]

TRAIN_CSV = "data/self_reported_ekman_zeroshot_preds.csv"

OUTPUT_DIR = "data/lora_emotion_token_baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
NUM_EPOCHS = 3
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LR = 2e-4
MAX_LENGTH = 1024
MAX_PER_CLASS = 400

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)


# ============================================================
# Helpers
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def model_safe_name(model_name):
    return model_name.replace("/", "-")


def prompt_template(text):
    return f"In this text: {text}, the emotion implied is:"


def build_emotion_token_map(tokenizer):
    emotion_to_token = {}

    print("Emotion target tokens:")
    for emotion in EMOTIONS:
        ids = tokenizer.encode(" " + emotion, add_special_tokens=False)
        token_id = int(ids[-1])
        emotion_to_token[emotion] = token_id
        print(emotion, ids, token_id, repr(tokenizer.decode([token_id])))

    if len(set(emotion_to_token.values())) != len(EMOTIONS):
        raise ValueError(f"Duplicate emotion token ids: {emotion_to_token}")

    return emotion_to_token


def get_final_token_logits(outputs, attention_mask):
    logits = outputs.logits
    mask = attention_mask.long()
    positions = torch.arange(mask.size(1), device=mask.device).unsqueeze(0)
    final_indices = (mask * positions).max(dim=1).values
    batch_indices = torch.arange(logits.size(0), device=logits.device)
    return logits[batch_indices, final_indices, :]


def get_lora_target_modules(model_name):
    if "gemma" in model_name.lower():
        return ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    if "llama" in model_name.lower():
        return ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    return ["q_proj", "v_proj"]


# ============================================================
# Data
# ============================================================

def load_main_data():
    df = pd.read_csv(TRAIN_CSV)
    df = df[["input_text", "emotion"]].dropna().copy()
    df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
    df = df[df["emotion"].isin(EMOTIONS)].copy()

    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=SEED,
        stratify=df["emotion"],
    )

    train_df = pd.concat(
        [
            g.sample(n=min(len(g), MAX_PER_CLASS), random_state=SEED)
            for _, g in train_df.groupby("emotion")
        ],
        ignore_index=True,
    )

    train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    return train_df, val_df


def load_xed():
    df = pd.read_csv("data/xed-en.csv")
    df = df[["text", "emotion"]].dropna().copy()
    df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
    df = df[df["emotion"].isin(EMOTIONS)].copy()
    df["input_text"] = df["text"].astype(str)
    return df[["input_text", "emotion"]].reset_index(drop=True)


def load_emoevent():
    df = pd.read_csv("data/emoevent_en.csv", sep="\t")
    df = df[df["emotion"] != "others"].copy()
    df = df[["tweet", "emotion"]].dropna().copy()
    df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
    df = df[df["emotion"].isin(EMOTIONS)].copy()
    df["input_text"] = df["tweet"].astype(str)
    return df[["input_text", "emotion"]].reset_index(drop=True)


def make_batches(df, batch_size, shuffle=False):
    if shuffle:
        df = df.sample(frac=1).reset_index(drop=True)

    for start in range(0, len(df), batch_size):
        yield df.iloc[start:start + batch_size]


# ============================================================
# Model
# ============================================================

def load_lora_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        device_map=DEVICE,
    )

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=get_lora_target_modules(model_name),
        bias="none",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    model.train()

    return model, tokenizer


# ============================================================
# Training
# ============================================================

def train_one_epoch(model, tokenizer, train_df, emotion_to_token, optimizer):
    model.train()
    losses = []

    emotion_to_idx = {e: i for i, e in enumerate(EMOTIONS)}
    target_token_ids = torch.tensor(
        [emotion_to_token[e] for e in EMOTIONS],
        device=DEVICE,
        dtype=torch.long,
    )

    for batch_df in tqdm(make_batches(train_df, BATCH_SIZE, shuffle=True), total=math.ceil(len(train_df) / BATCH_SIZE)):
        prompts = [prompt_template(x) for x in batch_df["input_text"].tolist()]
        labels = torch.tensor(
            [emotion_to_idx[e] for e in batch_df["emotion"].tolist()],
            device=DEVICE,
            dtype=torch.long,
        )

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        input_ids = inputs["input_ids"].to(DEVICE)
        attention_mask = inputs["attention_mask"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        final_logits = get_final_token_logits(outputs, attention_mask)
        emotion_logits = final_logits[:, target_token_ids]

        loss = F.cross_entropy(emotion_logits, labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(float(loss.detach().cpu()))

        del input_ids, attention_mask, outputs, final_logits, emotion_logits, loss, labels
        cleanup_memory()

    return float(np.mean(losses))


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_emotion_dataset(model, tokenizer, df, dataset_name, model_name, emotion_to_token):
    model.eval()

    target_token_ids = torch.tensor(
        [emotion_to_token[e] for e in EMOTIONS],
        device=DEVICE,
        dtype=torch.long,
    )

    y_true = []
    y_pred = []

    for batch_df in tqdm(make_batches(df, EVAL_BATCH_SIZE, shuffle=False), desc=f"eval {dataset_name}"):
        prompts = [prompt_template(x) for x in batch_df["input_text"].tolist()]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        input_ids = inputs["input_ids"].to(DEVICE)
        attention_mask = inputs["attention_mask"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        final_logits = get_final_token_logits(outputs, attention_mask)
        emotion_logits = final_logits[:, target_token_ids]

        preds = torch.argmax(emotion_logits, dim=-1).detach().cpu().tolist()

        y_pred.extend([EMOTIONS[i] for i in preds])
        y_true.extend(batch_df["emotion"].astype(str).str.lower().str.strip().tolist())

        del input_ids, attention_mask, outputs, final_logits, emotion_logits
        cleanup_memory()

    report = classification_report(
        y_true,
        y_pred,
        labels=EMOTIONS,
        output_dict=True,
        zero_division=0,
    )

    summary = {
        "model_name": model_name,
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=EMOTIONS, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=EMOTIONS, average="weighted", zero_division=0),
        "n": len(y_true),
    }

    per_emotion_rows = []
    for emotion in EMOTIONS:
        r = report[emotion]
        per_emotion_rows.append({
            "model_name": model_name,
            "dataset": dataset_name,
            "emotion": emotion,
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1-score"],
            "support": r["support"],
        })

    return summary, pd.DataFrame(per_emotion_rows)


@torch.no_grad()
def evaluate_general_ppl(model, tokenizer, max_texts=500, batch_size=4):
    model.eval()

    wiki = load_dataset(
        "wikitext",
        "wikitext-103-raw-v1",
        split="test",
        cache_dir=f"{CACHE_ROOT}/datasets",
    )

    texts = [t for t in wiki["text"] if len(str(t).strip()) > 100]
    texts = texts[:max_texts]

    total_loss = 0.0
    total_tokens = 0

    for start in tqdm(range(0, len(texts), batch_size), desc="general ppl"):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        input_ids = inputs["input_ids"].to(DEVICE)
        attention_mask = inputs["attention_mask"].to(DEVICE)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        num_tokens = int((labels != -100).sum().item())
        total_loss += float(outputs.loss.detach().cpu()) * num_tokens
        total_tokens += num_tokens

        del input_ids, attention_mask, labels, outputs
        cleanup_memory()

    avg_loss = total_loss / max(total_tokens, 1)
    return float(math.exp(avg_loss))


# ============================================================
# Run
# ============================================================

def run_one_model(model_name):
    safe_name = model_safe_name(model_name)
    out_dir = os.path.join(OUTPUT_DIR, safe_name)
    os.makedirs(out_dir, exist_ok=True)

    train_df, val_df = load_main_data()

    datasets = {
        "main_val": val_df,
        "xed_en": load_xed(),
        "emoevent_en": load_emoevent(),
    }

    model, tokenizer = load_lora_model(model_name)
    emotion_to_token = build_emotion_token_map(tokenizer)

    with open(os.path.join(out_dir, "emotion_token_map.json"), "w") as f:
        json.dump(emotion_to_token, f, indent=2)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.0,
    )

    all_epoch_rows = []
    all_per_emotion_rows = []

    best_macro = -1
    best_epoch = None
    best_dir = os.path.join(out_dir, "best_adapter")

    # Epoch 0 baseline with randomly initialized LoRA disabled by setting eval still includes zero LoRA effect initially.
    for epoch in range(0, NUM_EPOCHS + 1):
        if epoch > 0:
            train_loss = train_one_epoch(
                model=model,
                tokenizer=tokenizer,
                train_df=train_df,
                emotion_to_token=emotion_to_token,
                optimizer=optimizer,
            )
        else:
            train_loss = np.nan

        epoch_summaries = []

        for dataset_name, dataset_df in datasets.items():
            summary, per_emotion = evaluate_emotion_dataset(
                model=model,
                tokenizer=tokenizer,
                df=dataset_df,
                dataset_name=dataset_name,
                model_name=model_name,
                emotion_to_token=emotion_to_token,
            )

            summary["epoch"] = epoch
            summary["train_loss"] = train_loss
            epoch_summaries.append(summary)
            all_epoch_rows.append(summary)

            per_emotion["epoch"] = epoch
            all_per_emotion_rows.append(per_emotion)

        main_val_macro = [
            x["macro_f1"] for x in epoch_summaries
            if x["dataset"] == "main_val"
        ][0]

        if main_val_macro > best_macro:
            best_macro = main_val_macro
            best_epoch = epoch
            model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)

        pd.DataFrame(all_epoch_rows).to_csv(
            os.path.join(out_dir, "epoch_metrics_long.csv"),
            index=False,
        )

        pd.concat(all_per_emotion_rows, ignore_index=True).to_csv(
            os.path.join(out_dir, "epoch_per_emotion_f1.csv"),
            index=False,
        )

    general_ppl = evaluate_general_ppl(model, tokenizer)

    summary = {
        "model_name": model_name,
        "best_epoch": best_epoch,
        "best_main_val_macro_f1": best_macro,
        "general_ppl_final": general_ppl,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    del model
    del tokenizer
    cleanup_memory()

    return summary


def main():
    set_seed(SEED)

    summaries = []
    failures = []

    for model_name in MODELS:
        try:
            print("\n" + "=" * 100)
            print(model_name)
            print("=" * 100)

            summary = run_one_model(model_name)
            summaries.append(summary)

        except Exception as e:
            print(f"FAILED {model_name}: {repr(e)}")
            failures.append({"model_name": model_name, "error": repr(e)})
            cleanup_memory()

    pd.DataFrame(summaries).to_csv(
        os.path.join(OUTPUT_DIR, "all_model_summaries.csv"),
        index=False,
    )

    if failures:
        pd.DataFrame(failures).to_csv(
            os.path.join(OUTPUT_DIR, "failures.csv"),
            index=False,
        )


if __name__ == "__main__":
    main()